In [67]:
import os
import glob
import random
import shutil

import numpy as np
import matplotlib.pyplot as plt
import librosa
import soundfile as sf
import pyroomacoustics as pra
import joblib
import torch

from sklearn.preprocessing import StandardScaler

# ------------------------------------------------------------------
# 기본 경로
# ------------------------------------------------------------------
WAV_DIR = r'F:/datas/AID/AID/mic_dev'
NORM_PATH = 'expand_features'
WTS_PATH = os.path.join(NORM_PATH, 'scaler.joblib')

CHECKPOINT_PATH_BASELINE = r'./checkpoint/baseline.ckpt'
CHECKPOINT_PATH_MY15 = r'./checkpoint/my_model15.ckpt'

# ------------------------------------------------------------------
# 오디오 / 라벨 기본 설정
# ------------------------------------------------------------------
SR = 24000
SCENE_SEC = 10
SCENE_T = int(SCENE_SEC * SR)

LABEL_HOP_S = 0.1
HOP = 480
N_MELS = 64
N_MICS = 4
N_CHANNELS = 10   # 4 mel + 6 gcc
MODEL_N_CLASSES = 13

# ------------------------------------------------------------------
# 저장 폴더
# labels/dev_test 구조를 바로 사용
# ------------------------------------------------------------------
SIM_WAV_DIR = os.path.join(NORM_PATH, 'sim_wavs')
LABEL_ROOT_DIR = os.path.join(NORM_PATH, 'labels')
LABEL_DIR = os.path.join(LABEL_ROOT_DIR, 'dev_test')

EVAL_OUTPUT_ROOT = os.path.join(NORM_PATH, 'eval_compare')

os.makedirs(NORM_PATH, exist_ok=True)
os.makedirs(SIM_WAV_DIR, exist_ok=True)
os.makedirs(LABEL_ROOT_DIR, exist_ok=True)
os.makedirs(LABEL_DIR, exist_ok=True)
os.makedirs(EVAL_OUTPUT_ROOT, exist_ok=True)

# ------------------------------------------------------------------
# simulation parameters
# ------------------------------------------------------------------
room_dim = [8.0, 6.0, 3.0]

multi_source_positions = [
    np.array([2.0, 4.0, 1.5], dtype=np.float32),
    np.array([6.0, 2.0, 1.5], dtype=np.float32),
    np.array([4.0, 5.0, 2.0], dtype=np.float32),
]

mic_center = np.array([5.33, 3.0, 1.5], dtype=np.float32)
n_sources = 3

print('Cell 1 ready')
print('SIM_WAV_DIR :', SIM_WAV_DIR)
print('LABEL_DIR   :', LABEL_DIR)
print('WTS_PATH    :', WTS_PATH)

Cell 1 ready
SIM_WAV_DIR : expand_features\sim_wavs
LABEL_DIR   : expand_features\labels\dev_test
WTS_PATH    : expand_features\scaler.joblib


In [69]:
def scan_aid_classes(wav_dir):
    wav_files = glob.glob(os.path.join(wav_dir, '**', '*.wav'), recursive=True)
    if not wav_files:
        raise ValueError(f'WAV 파일을 찾지 못했습니다: {wav_dir}')

    class_names = sorted({
        os.path.basename(p).split('_')[0]
        for p in wav_files
    })
    return wav_files, class_names


all_wav_files, discovered_class_names = scan_aid_classes(WAV_DIR)

print("=== AID에서 발견된 클래스 이름들 ===")
print(discovered_class_names)
print(f"총 클래스 수: {len(discovered_class_names)}")


# ------------------------------------------------------------------
# 반드시 네 checkpoint 기준 13-class 매핑으로 직접 채워야 함
# key   : AID 파일명 prefix
# value : checkpoint 모델의 class idx (0~12)
# ------------------------------------------------------------------
TARGET_CLASS_MAP = {
    'tin_box': 0,
    'blender': 1,
    'cough': 2,
    'drill': 3,
    'whistle': 4,
    'clapping': 5,
    'footsteps': 6,
    'glass_jar': 7,
    'hairdryer': 8,
    'misc': 9,
    'music_box': 10,
    'plastic_box': 11,
    'vacuum': 12,
}


def validate_target_class_map(target_class_map, model_n_classes=13):
    if not target_class_map:
        raise ValueError("TARGET_CLASS_MAP이 비어 있습니다. 13-class 매핑을 채워주세요.")

    values = list(target_class_map.values())

    if len(set(values)) != len(values):
        raise ValueError(f"TARGET_CLASS_MAP에 중복 class idx가 있습니다: {values}")

    if min(values) < 0 or max(values) >= model_n_classes:
        raise ValueError(
            f"class idx 범위가 잘못되었습니다. "
            f"0 ~ {model_n_classes - 1} 범위여야 합니다. got={sorted(set(values))}"
        )

    if len(set(values)) != model_n_classes:
        raise ValueError(
            f"현재 매핑된 고유 class idx 수가 {model_n_classes}개가 아닙니다. "
            f"got={len(set(values))}, values={sorted(set(values))}"
        )


validate_target_class_map(TARGET_CLASS_MAP, MODEL_N_CLASSES)

mapped_wav_files = []
skipped_wav_files = []

for p in all_wav_files:
    cls = os.path.basename(p).split('_')[0]
    if cls in TARGET_CLASS_MAP:
        mapped_wav_files.append(p)
    else:
        skipped_wav_files.append(p)

print("\n=== TARGET_CLASS_MAP 기준 필터링 결과 ===")
print(f"사용할 wav 수 : {len(mapped_wav_files)}")
print(f"제외할 wav 수 : {len(skipped_wav_files)}")

used_class_names = sorted({
    os.path.basename(p).split('_')[0]
    for p in mapped_wav_files
})
print("실제로 사용되는 클래스:", used_class_names)

idx_to_target_name = {v: k for k, v in TARGET_CLASS_MAP.items()}
print("\n=== checkpoint 기준 class idx -> AID class name ===")
for i in range(MODEL_N_CLASSES):
    print(i, "->", idx_to_target_name[i])

=== AID에서 발견된 클래스 이름들 ===
['blender', 'book', 'breath', 'chest', 'clapping', 'cloth', 'clucking', 'consonant', 'cough', 'cutlery', 'drill', 'foot', 'glass', 'hairdryer', 'keys', 'lighter', 'lip', 'metal', 'misc', 'moka', 'music', 'paper', 'pen', 'plastic', 'plates', 'roll', 'rubber', 'scissors', 'shoe', 'snapping', 'tape', 'tin', 'tool', 'vacuum', 'whisk', 'whistle', 'zipper']
총 클래스 수: 37

=== TARGET_CLASS_MAP 기준 필터링 결과 ===
사용할 wav 수 : 150
제외할 wav 수 : 600
실제로 사용되는 클래스: ['blender', 'clapping', 'cough', 'drill', 'hairdryer', 'misc', 'vacuum', 'whistle']

=== checkpoint 기준 class idx -> AID class name ===
0 -> tin_box
1 -> blender
2 -> cough
3 -> drill
4 -> whistle
5 -> clapping
6 -> footsteps
7 -> glass_jar
8 -> hairdryer
9 -> misc
10 -> music_box
11 -> plastic_box
12 -> vacuum


## 1. 마이크 배열 좌표 정의 (ArrayGeometryFactory)

Tetrahedral / ULA / Circular 세 가지 4mic 배열의 3D 좌표를 정의합니다.

In [70]:
# =============================================================================
# 태스크 2.1: ArrayGeometryFactory 함수 구현
# Requirements: 1.1, 1.2, 1.3, 1.5, 8.4
# =============================================================================

def make_tetrahedral_array(r=0.042):
    """
    STARSS23 원본 정사면체 4mic 배열.
    구면 좌표(azimuth φ, elevation θ) → 직교 좌표 변환.
    x = r·cos(θ)·cos(φ),  y = r·cos(θ)·sin(φ),  z = r·sin(θ)
    반환: [3, 4]  (행=x/y/z, 열=마이크)
    """
    # (φ_deg, θ_deg) 순서: azimuth, elevation
    coords_deg = [
        ( 45,  35),   # M1
        (-45, -35),   # M2
        (135, -35),   # M3
        (-135,  35),  # M4
    ]
    n_mics = len(coords_deg)
    assert n_mics == 4, f'마이크 수는 4이어야 합니다. 현재: {n_mics}'
    pos = np.zeros((3, n_mics))
    for i, (phi_deg, theta_deg) in enumerate(coords_deg):
        phi   = np.deg2rad(phi_deg)
        theta = np.deg2rad(theta_deg)
        pos[0, i] = r * np.cos(theta) * np.cos(phi)   # x
        pos[1, i] = r * np.cos(theta) * np.sin(phi)   # y
        pos[2, i] = r * np.sin(theta)                  # z
    return pos


def make_ula_array(n_mics=4, spacing=0.06):
    """
    X축 등간격 선형 배열(ULA), 중심 원점.
    반환: [3, 4]
    """
    assert n_mics == 4, f'마이크 수는 4이어야 합니다. 현재: {n_mics}'
    xs = np.arange(n_mics) * spacing
    xs -= xs.mean()   # 중심 원점
    pos = np.zeros((3, n_mics))
    pos[0] = xs
    return pos


def make_circular_array(n_mics=4, r=0.035):
    """
    XY 평면 등간격 원형 배열, 중심 원점.
    반환: [3, 4]
    """
    assert n_mics == 4, f'마이크 수는 4이어야 합니다. 현재: {n_mics}'
    angles = np.linspace(0, 2 * np.pi, n_mics, endpoint=False)
    pos = np.zeros((3, n_mics))
    pos[0] = r * np.cos(angles)   # x
    pos[1] = r * np.sin(angles)   # y
    return pos


def compute_inter_mic_distances(mic_pos):
    """
    모든 마이크 쌍 간 유클리드 거리 계산.
    입력: [3, M]
    반환: [M*(M-1)/2] 거리 배열 (단위: m)
    """
    M = mic_pos.shape[1]
    dists = []
    for i in range(M):
        for j in range(i + 1, M):
            d = np.linalg.norm(mic_pos[:, i] - mic_pos[:, j])
            dists.append(d)
    return np.array(dists)


# --- ARRAY_CONFIGS 딕셔너리 (Requirements 8.4) ---
ARRAY_CONFIGS = {
    'tetrahedral': make_tetrahedral_array(r=0.042),
    'ula':         make_ula_array(n_mics=4, spacing=0.06),
    'circular':    make_circular_array(n_mics=4, r=0.04),
}

print('=== 배열 좌표 및 마이크 간 거리 ===')
for name, pos in ARRAY_CONFIGS.items():
    dists = compute_inter_mic_distances(pos)
    print(f'[{name}]  shape={pos.shape}')
    print(f'  InterMicDistances (m): {np.round(dists, 4)}')
    print(f'  min={dists.min()*100:.2f}cm  max={dists.max()*100:.2f}cm')

=== 배열 좌표 및 마이크 간 거리 ===
[tetrahedral]  shape=(3, 4)
  InterMicDistances (m): [0.0685 0.0685 0.0688 0.0688 0.0685 0.0685]
  min=6.85cm  max=6.88cm
[ula]  shape=(3, 4)
  InterMicDistances (m): [0.06 0.12 0.18 0.06 0.12 0.06]
  min=6.00cm  max=18.00cm
[circular]  shape=(3, 4)
  InterMicDistances (m): [0.0566 0.08   0.0566 0.0566 0.08   0.0566]
  min=5.66cm  max=8.00cm


In [71]:
# =============================================================================
# 태스크 2.5: 단위 테스트 — 배열 좌표 검증
# Requirements: 1.1, 1.4, 8.4
# =============================================================================

# (1) Tetrahedral M1 좌표가 STARSS23 명세와 일치하는지 확인 (Requirements 1.1)
r = 0.042
phi, theta = np.deg2rad(45), np.deg2rad(35)
expected_m1 = np.array([
    r * np.cos(theta) * np.cos(phi),
    r * np.cos(theta) * np.sin(phi),
    r * np.sin(theta),
])
actual_m1 = ARRAY_CONFIGS['tetrahedral'][:, 0]
assert np.allclose(actual_m1, expected_m1, atol=1e-6), \
    f'M1 좌표 불일치: {actual_m1} vs {expected_m1}'
print('[PASS] Tetrahedral M1 좌표 STARSS23 명세 일치')

# (2) 세 배열 모두 최소 InterMicDistance >= 5cm 확인 (Requirements 1.4)
for name, pos in ARRAY_CONFIGS.items():
    dists = compute_inter_mic_distances(pos)
    assert dists.min() >= 0.05, \
        f'[{name}] 최소 InterMicDistance {dists.min()*100:.2f}cm < 5cm'
    print(f'[PASS] {name}: 최소 InterMicDistance = {dists.min()*100:.2f}cm >= 5cm')

# (3) ARRAY_CONFIGS 키 확인 (Requirements 8.4)
for key in ['tetrahedral', 'ula', 'circular']:
    assert key in ARRAY_CONFIGS, f'ARRAY_CONFIGS에 {key} 키 없음'
print('[PASS] ARRAY_CONFIGS 키 확인: tetrahedral, ula, circular')

print('\n=== 배열 좌표 단위 테스트 전체 통과 ===')

[PASS] Tetrahedral M1 좌표 STARSS23 명세 일치
[PASS] tetrahedral: 최소 InterMicDistance = 6.85cm >= 5cm
[PASS] ula: 최소 InterMicDistance = 6.00cm >= 5cm
[PASS] circular: 최소 InterMicDistance = 5.66cm >= 5cm
[PASS] ARRAY_CONFIGS 키 확인: tetrahedral, ula, circular

=== 배열 좌표 단위 테스트 전체 통과 ===


## 2. 룸 어쿠스틱 시뮬레이터 (RoomSimulator)

In [72]:
def simulate_room(
    dry_wavs,         # list of 1D np.ndarray
    src_positions,    # list of [x, y, z]
    mic_pos,          # np.ndarray, shape [3, M]
    sr=SR,
    rt60=0.3,
    room_dim=None,
    plot_room=False,
    source_meta=None,
    label_mic_center=None,
    distance_unit='cm',
    return_metadata=False,
):
    """
    여러 음원을 pyroomacoustics ShoeBox에서 시뮬레이션

    Returns
    -------
    signals : np.ndarray, shape [M, T_sim]
    metadata : list[dict] (return_metadata=True일 때)
    """
    if room_dim is None:
        room_dim = [8.0, 6.0, 3.0]

    if not (len(dry_wavs) == len(src_positions)):
        raise ValueError("dry_wavs, src_positions 길이가 같아야 합니다.")

    e_absorption, max_order = pra.inverse_sabine(rt60, room_dim)
    room = pra.ShoeBox(
        room_dim,
        fs=sr,
        materials=pra.Material(e_absorption),
        max_order=max_order,
    )

    for idx, (wav, pos) in enumerate(zip(dry_wavs, src_positions)):
        wav = np.asarray(wav, dtype=np.float64)
        if wav.ndim != 1:
            raise ValueError(f"음원 {idx}는 mono 1차원 배열이어야 합니다. got shape={wav.shape}")

        for axis, (s, d) in enumerate(zip(pos, room_dim)):
            if not (0 < s < d):
                raise RuntimeError(
                    f"음원 {idx} 위치가 룸 경계 밖입니다. "
                    f"axis={axis}: src={s:.3f}, room=[0, {d}]"
                )

        room.add_source(pos, signal=wav)

    mic_pos = np.asarray(mic_pos, dtype=np.float64)
    if mic_pos.ndim != 2 or mic_pos.shape[0] != 3:
        raise ValueError(f"mic_pos shape은 [3, M] 이어야 합니다. got {mic_pos.shape}")

    room.add_microphone_array(mic_pos)

    if plot_room:
        fig, ax = room.plot()
        fig.set_size_inches(8, 6)
        ax.set_xlim([0, room_dim[0]])
        ax.set_ylim([0, room_dim[1]])
        ax.set_zlim([0, room_dim[2]])
        ax.set_box_aspect([room_dim[0], room_dim[1], room_dim[2]])
        ax.set_xlabel("X")
        ax.set_ylabel("Y")
        ax.set_zlabel("Z")
        ax.set_title("SELD Room Simulation")
        plt.show()

    room.simulate()
    signals = room.mic_array.signals.astype(np.float32)

    if not return_metadata:
        return signals

    if label_mic_center is None:
        label_mic_center = np.mean(mic_pos, axis=1)

    metadata = []
    for i, pos in enumerate(src_positions):
        src_pos = np.asarray(pos, dtype=np.float64).reshape(3)
        mic_center_ = np.asarray(label_mic_center, dtype=np.float64).reshape(3)

        rel = src_pos - mic_center_
        x, y, z = rel
        horiz = np.sqrt(x**2 + y**2)
        dist_m = np.sqrt(x**2 + y**2 + z**2)

        azimuth = int(np.round(np.degrees(np.arctan2(y, x))))
        elevation = int(np.round(np.degrees(np.arctan2(z, horiz))))

        if distance_unit == 'm':
            distance = int(np.round(dist_m))
        elif distance_unit == 'cm':
            distance = int(np.round(dist_m * 100.0))
        else:
            raise ValueError("distance_unit must be 'm' or 'cm'")

        meta = {
            'source_idx': i,
            'src_position': src_pos.astype(np.float32),
            'azimuth': azimuth,
            'elevation': elevation,
            'distance': distance,
        }

        if source_meta is not None:
            meta.update(source_meta[i])

        metadata.append(meta)

    return signals, metadata

In [73]:
from utils.cls_tools.cls_feature_class import FeatureClass

def extract_class_name_from_path(file_path):
    base = os.path.basename(file_path)
    return base.split('_')[0]

def make_scene_stem(sim_idx, fold_id=4, room_id=24):
    """
    예:
    sim_idx=0 -> fold4_room24_mix000
    """
    return f"fold{fold_id}_room{room_id}_mix{sim_idx:03d}"

def samples_to_label_frame_range(start_sample, n_samples, sr, label_hop_s=0.1):
    label_hop = int(round(sr * label_hop_s))
    if n_samples <= 0:
        return None, None

    start_frame = start_sample // label_hop
    end_sample_inclusive = start_sample + n_samples - 1
    end_frame = end_sample_inclusive // label_hop
    return int(start_frame), int(end_frame)

def make_starss_rows_for_event(
    class_idx,
    source_idx,
    azimuth,
    elevation,
    distance,
    start_sample,
    n_active_samples,
    sr,
    label_hop_s=0.1,
):
    start_frame, end_frame = samples_to_label_frame_range(
        start_sample=start_sample,
        n_samples=n_active_samples,
        sr=sr,
        label_hop_s=label_hop_s,
    )

    if start_frame is None:
        return []

    rows = []
    for frame_idx in range(start_frame, end_frame + 1):
        rows.append([
            int(frame_idx),
            int(class_idx),
            int(source_idx),
            int(azimuth),
            int(elevation),
            int(distance),
        ])
    return rows

def save_starss_label_csv(label_rows, csv_path):
    label_rows = sorted(label_rows, key=lambda x: (x[0], x[2], x[1]))
    arr = np.asarray(label_rows, dtype=np.int32)
    np.savetxt(csv_path, arr, fmt='%d', delimiter=',')

def build_scene_and_labels(
    wav_paths,
    src_positions,
    mic_pos,
    target_class_map,
    sr=SR,
    scene_t=SCENE_T,
    rt60=0.2,
    room_dim=None,
    label_hop_s=0.1,
    distance_unit='cm',
    random_crop_if_long=True,
    use_reverb_tail_for_label=False,
):
    """
    여러 AID wav를 하나의 scene으로 합성하고,
    STARSS23 형식 라벨 rows를 생성합니다.
    """
    if room_dim is None:
        room_dim = [8.0, 6.0, 3.0]

    if len(wav_paths) != len(src_positions):
        raise ValueError("wav_paths와 src_positions 길이는 같아야 합니다.")

    scene = np.zeros((mic_pos.shape[1], scene_t), dtype=np.float32)
    label_rows = []
    event_infos = []

    label_mic_center = np.mean(mic_pos, axis=1)

    for source_idx, (wav_path, src_pos) in enumerate(zip(wav_paths, src_positions)):
        class_name = extract_class_name_from_path(wav_path)
        if class_name not in target_class_map:
            raise KeyError(f"TARGET_CLASS_MAP에 없는 클래스입니다: {class_name}")

        class_idx = target_class_map[class_name]

        dry, _ = librosa.load(wav_path, sr=sr, mono=True)
        dry = dry.astype(np.float32)

        if len(dry) > scene_t and random_crop_if_long:
            crop_start = random.randint(0, len(dry) - scene_t)
            dry = dry[crop_start:crop_start + scene_t]

        dry_active_len = len(dry)

        sim_sig, sim_meta = simulate_room(
            dry_wavs=[dry],
            src_positions=[src_pos],
            mic_pos=mic_pos,
            sr=sr,
            rt60=rt60,
            room_dim=room_dim,
            plot_room=False,
            source_meta=[{
                'class_name': class_name,
                'class_idx': class_idx,
                'wav_path': wav_path,
                'source_idx': source_idx,
            }],
            label_mic_center=label_mic_center,
            distance_unit=distance_unit,
            return_metadata=True,
        )

        meta = sim_meta[0]
        sim_len = sim_sig.shape[1]

        if sim_len >= scene_t:
            start_idx = random.randint(0, sim_len - scene_t)
            scene += sim_sig[:, start_idx:start_idx + scene_t]
            placed_start_sample = 0
            label_active_len = scene_t if use_reverb_tail_for_label else min(dry_active_len, scene_t)
        else:
            onset = random.randint(0, scene_t - sim_len)
            scene[:, onset:onset + sim_len] += sim_sig
            placed_start_sample = onset
            label_active_len = sim_len if use_reverb_tail_for_label else dry_active_len

        rows = make_starss_rows_for_event(
            class_idx=meta['class_idx'],
            source_idx=meta['source_idx'],
            azimuth=meta['azimuth'],
            elevation=meta['elevation'],
            distance=meta['distance'],
            start_sample=placed_start_sample,
            n_active_samples=label_active_len,
            sr=sr,
            label_hop_s=label_hop_s,
        )
        label_rows.extend(rows)

        event_infos.append({
            'wav_path': wav_path,
            'class_name': class_name,
            'class_idx': class_idx,
            'source_idx': source_idx,
            'src_position': np.asarray(src_pos, dtype=np.float32),
            'azimuth': meta['azimuth'],
            'elevation': meta['elevation'],
            'distance': meta['distance'],
            'placed_start_sample': placed_start_sample,
            'label_active_len': label_active_len,
        })

    peak = np.max(np.abs(scene))
    if peak > 1.0:
        scene = scene / (peak + 1e-8)

    scene = np.clip(scene, -1.0, 1.0)

    return scene, label_rows, event_infos

print('helper functions ready')

helper functions ready


## 3. Baseline 전처리 파이프라인 (BaselinePreprocessor)

고정 mel + GCC-PHAT → `[10, Frames, 64]`

In [74]:
# ------------------------------------------------------------------
# 기존 생성물 정리
# ------------------------------------------------------------------
for p in glob.glob(os.path.join(SIM_WAV_DIR, '*.wav')):
    os.remove(p)

for p in glob.glob(os.path.join(LABEL_DIR, '*.csv')):
    os.remove(p)

print("기존 sim wav / label csv 삭제 완료")


# ------------------------------------------------------------------
# feature extractor / scaler 준비
# ------------------------------------------------------------------
params = {
    'dataset_dir': WAV_DIR,
    'feat_label_dir': NORM_PATH,
    'dataset': 'mic',
    'fs': SR,
    'hop_len_s': HOP / SR,
    'label_hop_len_s': LABEL_HOP_S,
    'nb_mel_bins': N_MELS,
    'unique_classes': MODEL_N_CLASSES,
    'spatial_map': False,
    'multi_accdoa': False,
    'use_salsalite': False,
    'raw_chunks': False,
    'saved_chunks': False,
}

feat_cls = FeatureClass(params, is_eval=True)
scaler = StandardScaler()

raw_files = mapped_wav_files[:]
if len(raw_files) < n_sources:
    raise ValueError(f"[오류] 매핑된 원본 파일이 {n_sources}개 미만입니다. ({len(raw_files)}개 발견)")

random.shuffle(raw_files)

file_groups = [
    raw_files[i:i+n_sources]
    for i in range(0, len(raw_files) - len(raw_files) % n_sources, n_sources)
]

NUM_SIMULATIONS = len(file_groups)

print(f"=== 시뮬레이션 데이터 + STARSS23 형식 라벨 생성 시작 ===")
print(f"총 {NUM_SIMULATIONS}개의 scene을 생성합니다...\n")

for sim_idx, selected_files in enumerate(file_groups):
    rt60 = 0.2
    mic_pos = ARRAY_CONFIGS['tetrahedral'] + mic_center[:, None]

    scene, label_rows, event_infos = build_scene_and_labels(
        wav_paths=selected_files,
        src_positions=multi_source_positions,
        mic_pos=mic_pos,
        target_class_map=TARGET_CLASS_MAP,
        sr=SR,
        scene_t=SCENE_T,
        rt60=rt60,
        room_dim=room_dim,
        label_hop_s=LABEL_HOP_S,
        distance_unit='cm',
        use_reverb_tail_for_label=False,
    )

    scene_stem = make_scene_stem(sim_idx, fold_id=4, room_id=24)

    wav_save_path = os.path.join(SIM_WAV_DIR, f'{scene_stem}.wav')
    label_csv_path = os.path.join(LABEL_DIR, f'{scene_stem}.csv')

    sf.write(wav_save_path, scene.T, SR)
    save_starss_label_csv(label_rows, label_csv_path)

    # scaler 업데이트용 baseline feature 생성
    audio_in = scene.T + feat_cls._eps
    nb_feat_frames = int(SCENE_T / float(feat_cls._hop_len))

    spectra = feat_cls._spectrogram(audio_in, nb_feat_frames)
    mel_spect = feat_cls._get_mel_spectrogram(spectra)
    gcc = feat_cls._get_gcc(spectra)
    feat = np.concatenate((mel_spect, gcc), axis=-1)

    scaler.partial_fit(feat)

    if (sim_idx + 1) % 20 == 0 or (sim_idx + 1) == NUM_SIMULATIONS:
        print(f"[{sim_idx + 1}/{NUM_SIMULATIONS}] 완료")
        print(f"  wav   : {wav_save_path}")
        print(f"  label : {label_csv_path}")
        print(f"  events: {len(event_infos)}")

joblib.dump(scaler, WTS_PATH)

print(f"\n[성공] 시뮬레이션 기반 scaler 저장 완료")
print(f"WTS_PATH   : {WTS_PATH}")
print(f"SIM_WAV_DIR: {SIM_WAV_DIR}")
print(f"LABEL_DIR  : {LABEL_DIR}")

기존 sim wav / label csv 삭제 완료
=== 시뮬레이션 데이터 + STARSS23 형식 라벨 생성 시작 ===
총 50개의 scene을 생성합니다...

[20/50] 완료
  wav   : expand_features\sim_wavs\fold4_room24_mix019.wav
  label : expand_features\labels\dev_test\fold4_room24_mix019.csv
  events: 3
[40/50] 완료
  wav   : expand_features\sim_wavs\fold4_room24_mix039.wav
  label : expand_features\labels\dev_test\fold4_room24_mix039.csv
  events: 3
[50/50] 완료
  wav   : expand_features\sim_wavs\fold4_room24_mix049.wav
  label : expand_features\labels\dev_test\fold4_room24_mix049.csv
  events: 3

[성공] 시뮬레이션 기반 scaler 저장 완료
WTS_PATH   : expand_features\scaler.joblib
SIM_WAV_DIR: expand_features\sim_wavs
LABEL_DIR  : expand_features\labels\dev_test


In [77]:
import os
import glob
import shutil
import numpy as np
import soundfile as sf
import joblib
import torch

from torch.utils.data import Dataset, DataLoader

from models.my_model15 import ResnetConformer_seddoa_nopool_2023 as MyModel15
from models.resnet_conformer_audio import ResnetConformer_seddoa_nopool_2023 as BaselineModel

from utils.write_csv import write_output_format_file
from utils.cls_tools.cls_compute_seld_results import ComputeSELDResults
from utils.cls_tools.cls_feature_class import FeatureClass


DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

GT_EVAL_ROOT = os.path.join(NORM_PATH, 'labels_eval_nodist')
GT_EVAL_DIR = os.path.join(GT_EVAL_ROOT, 'dev_test')


def make_nodist_label_root(src_root, dst_root):
    """
    src_root 아래의 csv를 재귀적으로 읽어서,
    6열이면 마지막 distance 제거 후 dst_root에 같은 상대경로로 저장.
    5열이면 그대로 복사.
    """
    if os.path.exists(dst_root):
        shutil.rmtree(dst_root)
    os.makedirs(dst_root, exist_ok=True)

    src_csvs = glob.glob(os.path.join(src_root, '**', '*.csv'), recursive=True)
    if not src_csvs:
        raise ValueError(f'GT csv가 없습니다: {src_root}')

    for src_csv in src_csvs:
        rel_path = os.path.relpath(src_csv, src_root)
        dst_csv = os.path.join(dst_root, rel_path)
        os.makedirs(os.path.dirname(dst_csv), exist_ok=True)

        arr = np.loadtxt(src_csv, delimiter=',', ndmin=2)

        if arr.size == 0:
            np.savetxt(dst_csv, arr, fmt='%d', delimiter=',')
            continue

        if arr.shape[1] == 6:
            arr = arr[:, :5]
        elif arr.shape[1] == 5:
            pass
        else:
            raise ValueError(f'지원하지 않는 GT csv shape: {src_csv}, {arr.shape}')

        np.savetxt(dst_csv, arr.astype(np.int32), fmt='%d', delimiter=',')

    print(f'[OK] no-distance GT 생성 완료: {dst_root}')


def load_ckpt_state(ckpt_path, map_location='cpu'):
    ckpt = torch.load(ckpt_path, map_location=map_location)

    if isinstance(ckpt, dict):
        if 'model_state_dict' in ckpt:
            state = ckpt['model_state_dict']
        elif 'state_dict' in ckpt:
            state = ckpt['state_dict']
        else:
            state = ckpt
    else:
        state = ckpt

    new_state = {}
    for k, v in state.items():
        if k.startswith('module.'):
            new_state[k[len('module.'):]] = v
        else:
            new_state[k] = v
    return new_state


def pad_or_crop_multich_wav(x, target_len):
    """
    x: [C, T]
    """
    if x.shape[1] > target_len:
        x = x[:, :target_len]
    elif x.shape[1] < target_len:
        pad_len = target_len - x.shape[1]
        x = np.pad(x, ((0, 0), (0, pad_len)), mode='constant')
    return x


def load_multich_wav(wav_path, target_sr=SR, target_len=SCENE_T):
    """
    저장된 sim wav 읽기.
    저장 시 scene.T 로 저장했으므로 보통 [T, C] -> [C, T] 로 transpose.
    """
    x, sr = sf.read(wav_path)
    if sr != target_sr:
        raise ValueError(f"샘플레이트 불일치: {wav_path}, got={sr}, expected={target_sr}")

    x = np.asarray(x, dtype=np.float32)

    if x.ndim == 1:
        raise ValueError(f"multichannel wav가 아닙니다: {wav_path}, shape={x.shape}")

    # [T, C] -> [C, T]
    if x.shape[0] > x.shape[1]:
        x = x.T

    if x.shape[0] != N_MICS:
        raise ValueError(f"마이크 채널 수 불일치: {wav_path}, got={x.shape[0]}, expected={N_MICS}")

    x = pad_or_crop_multich_wav(x, target_len)
    return x


def build_feature_extractor():
    params = {
        'dataset_dir': SIM_WAV_DIR,
        'feat_label_dir': NORM_PATH,
        'dataset': 'mic',
        'fs': SR,
        'hop_len_s': HOP / SR,
        'label_hop_len_s': LABEL_HOP_S,
        'nb_mel_bins': N_MELS,
        'unique_classes': MODEL_N_CLASSES,
        'spatial_map': False,
        'multi_accdoa': False,
        'use_salsalite': False,
        'raw_chunks': False,
        'saved_chunks': False,
    }
    return FeatureClass(params, is_eval=True)


def baseline_preprocess_from_waveform(mic_signals, feat_cls, scaler):
    """
    mic_signals: [4, T]
    return: [10, frames, 64]
    """
    audio_in = mic_signals.T.astype(np.float32)  # [T, 4]
    audio_in = audio_in + feat_cls._eps

    nb_feat_frames = int(mic_signals.shape[1] / float(feat_cls._hop_len))

    spectra = feat_cls._spectrogram(audio_in, nb_feat_frames)
    mel_spect = feat_cls._get_mel_spectrogram(spectra)   # [frames, 4*64]
    gcc = feat_cls._get_gcc(spectra)                     # [frames, 6*64]

    feat = np.concatenate((mel_spect, gcc), axis=-1)    # [frames, 640]

    expected_dim = N_CHANNELS * N_MELS
    if feat.shape[1] != expected_dim:
        raise ValueError(f'baseline feature dim mismatch: got {feat.shape[1]}, expected {expected_dim}')

    feat = scaler.transform(feat)
    feat = feat.reshape(feat.shape[0], N_CHANNELS, N_MELS)   # [frames, 10, 64]
    feat = np.transpose(feat, (1, 0, 2))                     # [10, frames, 64]

    return torch.from_numpy(feat.astype(np.float32))


def raw_preprocess_for_mymodel(mic_signals):
    """
    mic_signals: [4, T]
    return: [4, n_frames, HOP]

    네 my_model(frontend 포함)이 raw framed waveform을 받는 구조 기준.
    만약 네 forward가 [B, 4, T]를 기대하면 여기만 바꾸면 됨.
    """
    T = mic_signals.shape[1]
    n_frames = T // HOP
    usable = n_frames * HOP
    wav = mic_signals[:, :usable].reshape(N_MICS, n_frames, HOP)
    return torch.from_numpy(wav.astype(np.float32))


def inspect_model_output(name, out):
    print(f'\n[{name}] type = {type(out)}')

    if torch.is_tensor(out):
        print(f'  tensor shape: {tuple(out.shape)}')

    elif isinstance(out, (list, tuple)):
        print(f'  len = {len(out)}')
        for i, x in enumerate(out):
            if torch.is_tensor(x):
                print(f'  [{i}] tensor shape: {tuple(x.shape)}')
            else:
                print(f'  [{i}] type: {type(x)}')

    elif isinstance(out, dict):
        print(f'  keys = {list(out.keys())}')
        for k, v in out.items():
            if torch.is_tensor(v):
                print(f'  {k}: {tuple(v.shape)}')
            else:
                print(f'  {k}: {type(v)}')


def pack_model_output_for_seddoa52(model_out, n_classes=13):
    """
    모델 출력을 [B, T, 52] = [B, T, 13 + 39] 로 맞춤.
    """
    feat_dim = 4 * n_classes  # 13 sed + 39 doa = 52

    def to_btf(x, feat_dim, name='tensor'):
        if x.ndim != 3:
            raise ValueError(f'{name} ndim must be 3, got shape={tuple(x.shape)}')

        if x.shape[-1] == feat_dim:
            return x.contiguous()

        if x.shape[1] == feat_dim:
            return x.permute(0, 2, 1).contiguous()

        raise ValueError(
            f'{name} shape mismatch. expected feat_dim={feat_dim}, got shape={tuple(x.shape)}'
        )

    if isinstance(model_out, dict):
        for k in ['pred', 'output', 'seddoa', 'logits']:
            if k in model_out and torch.is_tensor(model_out[k]):
                return to_btf(model_out[k], feat_dim, k)
        raise ValueError(f'dict output에서 52차원 pred를 찾지 못했습니다. keys={list(model_out.keys())}')

    if isinstance(model_out, (tuple, list)):
        for i, x in enumerate(model_out):
            if torch.is_tensor(x):
                try:
                    return to_btf(x, feat_dim, f'output[{i}]')
                except Exception:
                    pass
        raise ValueError('tuple/list output 안에서 52차원 pred tensor를 찾지 못했습니다.')

    if torch.is_tensor(model_out):
        return to_btf(model_out, feat_dim, 'output')

    raise ValueError(f'지원하지 않는 output type: {type(model_out)}')


def seddoa52_tensor_to_output_dict(
    pred,
    n_classes=13,
    sed_threshold=0.5,
    apply_sigmoid_to_sed=False,
):
    """
    pred: [T, 52]
    앞 13차원 = SED
    뒤 39차원 = DOA(x,y,z)

    반환:
        write_output_format_file용 dict
        out_dict[frame_idx] = [[class_idx, x, y, z], ...]
    """
    pred = np.asarray(pred)

    if pred.ndim != 2 or pred.shape[1] != 4 * n_classes:
        raise ValueError(f'pred shape must be [T, {4*n_classes}], got {pred.shape}')

    sed = pred[:, :n_classes]
    doa = pred[:, n_classes:]

    if apply_sigmoid_to_sed:
        sed = 1.0 / (1.0 + np.exp(-sed))

    x = doa[:, :n_classes]
    y = doa[:, n_classes:2*n_classes]
    z = doa[:, 2*n_classes:3*n_classes]

    out_dict = {}
    for t in range(pred.shape[0]):
        frame_events = []
        for c in range(n_classes):
            if sed[t, c] > sed_threshold:
                frame_events.append([c, x[t, c], y[t, c], z[t, c]])
        if frame_events:
            out_dict[t] = frame_events

    return out_dict


def print_seld_metrics(tag, metrics_tuple):
    print(f'\n[{tag}]')

    if len(metrics_tuple) >= 5:
        ER = metrics_tuple[0]
        F = metrics_tuple[1]
        LE = metrics_tuple[2]
        LR = metrics_tuple[3]
        SELD = metrics_tuple[4]

        print(f'ER        : {ER:.4f}')
        print(f'F-score   : {F:.4f}')
        print(f'LE        : {LE:.4f}')
        print(f'LR        : {LR:.4f}')
        print(f'SELD score: {SELD:.4f}')
    else:
        print(metrics_tuple)


class SimulatedSELDComparisonDataset(Dataset):
    """
    sim_wavs/*.wav 와 labels_root/**.csv 의 stem을 맞춰서 읽음
    baseline_input: [10, frames, 64]
    raw_input     : [4, n_frames, HOP]
    wav_name      : stem (예: fold4_room24_mix000)
    """
    def __init__(self, sim_wav_dir, label_root_dir, scaler_path):
        self.sim_wav_dir = sim_wav_dir
        self.label_root_dir = label_root_dir

        self.feat_cls = build_feature_extractor()
        self.scaler = joblib.load(scaler_path)

        wav_map = {
            os.path.splitext(os.path.basename(p))[0]: p
            for p in glob.glob(os.path.join(sim_wav_dir, '*.wav'))
        }

        label_paths = glob.glob(os.path.join(label_root_dir, '**', '*.csv'), recursive=True)
        label_map = {}
        for p in label_paths:
            stem = os.path.splitext(os.path.basename(p))[0]
            if stem in label_map:
                raise ValueError(f'중복 stem 발견: {stem}\n- {label_map[stem]}\n- {p}')
            label_map[stem] = p

        common_stems = sorted(set(wav_map.keys()) & set(label_map.keys()))
        if not common_stems:
            raise ValueError(
                f"공통 stem이 없습니다.\n"
                f"sim_wav_dir={sim_wav_dir}\n"
                f"label_root_dir={label_root_dir}\n"
                f"wav stems example={list(wav_map.keys())[:5]}\n"
                f"label stems example={list(label_map.keys())[:5]}"
            )

        self.items = [(stem, wav_map[stem], label_map[stem]) for stem in common_stems]
        print(f'[Dataset] matched items: {len(self.items)}')
        print(f'[Dataset] first few stems: {[x[0] for x in self.items[:5]]}')

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        stem, wav_path, label_path = self.items[idx]

        mic_signals = load_multich_wav(
            wav_path,
            target_sr=SR,
            target_len=SCENE_T,
        )  # [4, T]

        baseline_input = baseline_preprocess_from_waveform(
            mic_signals=mic_signals,
            feat_cls=self.feat_cls,
            scaler=self.scaler,
        )  # [10, frames, 64]

        raw_input = raw_preprocess_for_mymodel(
            mic_signals=mic_signals
        )  # [4, n_frames, HOP]

        return {
            'wav_name': stem,
            'wav_path': wav_path,
            'label_path': label_path,
            'baseline_input': baseline_input,
            'raw_input': raw_input,
        }


def build_models_for_comparison(
    baseline_ckpt_path,
    mymodel_ckpt_path,
    norm_path,
    device=DEVICE,
):
    baseline_model = BaselineModel(
        in_channel=N_CHANNELS,
        in_dim=N_MELS,
        out_dim=39,
    ).to(device)

    my_model = MyModel15(
        in_channel=N_CHANNELS,
        in_dim=N_MELS,
        out_dim=39,
        norm_path=norm_path,
    ).to(device)

    baseline_state = load_ckpt_state(baseline_ckpt_path, map_location=device)
    mymodel_state = load_ckpt_state(mymodel_ckpt_path, map_location=device)

    baseline_model.load_state_dict(baseline_state, strict=True)
    my_model.load_state_dict(mymodel_state, strict=True)

    baseline_model.eval()
    my_model.eval()

    print(f'[OK] baseline ckpt loaded: {baseline_ckpt_path}')
    print(f'[OK] my_model ckpt loaded: {mymodel_ckpt_path}')

    return baseline_model, my_model


def evaluate_baseline_vs_mymodel(
    sim_wav_dir=SIM_WAV_DIR,
    label_root_dir=LABEL_ROOT_DIR,
    baseline_ckpt_path=CHECKPOINT_PATH_BASELINE,
    mymodel_ckpt_path=CHECKPOINT_PATH_MY15,
    scaler_path=WTS_PATH,
    batch_size=1,
    num_workers=0,
    output_root=EVAL_OUTPUT_ROOT,
    sed_threshold=0.5,
    apply_sigmoid_to_sed=False,
    inspect_first_batch=True,
):
    # 1) GT 6열 -> old evaluator용 no-distance GT 생성
    make_nodist_label_root(label_root_dir, GT_EVAL_ROOT)

    # 2) dataset
    dataset = SimulatedSELDComparisonDataset(
        sim_wav_dir=sim_wav_dir,
        label_root_dir=GT_EVAL_ROOT,
        scaler_path=scaler_path,
    )

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )

    # 3) prediction dir 초기화
    baseline_pred_dir = os.path.join(output_root, 'baseline_preds')
    mymodel_pred_dir = os.path.join(output_root, 'mymodel_preds')

    for d in [baseline_pred_dir, mymodel_pred_dir]:
        if os.path.exists(d):
            shutil.rmtree(d)
        os.makedirs(d, exist_ok=True)

    # 4) model load
    baseline_model, my_model = build_models_for_comparison(
        baseline_ckpt_path="c:/26'interspeech/Solution_on_3D_SELD/output/SED-DOA_FOA/checkpoints/checkpoint_epoch357_step97461.pth",
        mymodel_ckpt_path="c:/26'interspeech/Monaural_SELD/output/my_experiment15/checkpoints/checkpoint_epoch337_step92001.pth",
        norm_path=scaler_path,
        device=DEVICE,
    )

    # 5) inference + prediction csv 저장
    print('\n>>> inference + prediction csv 저장 시작')
    with torch.no_grad():
        for batch_idx, batch in enumerate(loader):
            wav_names = batch['wav_name']

            baseline_input = batch['baseline_input'].to(DEVICE)   # [B, 10, frames, 64]
            raw_input = batch['raw_input'].to(DEVICE)             # [B, 4, n_frames, HOP]

            raw_baseline_out = baseline_model(baseline_input)
            raw_mymodel_out = my_model(raw_input)

            if inspect_first_batch and batch_idx == 0:
                inspect_model_output('baseline raw output', raw_baseline_out)
                inspect_model_output('my_model raw output', raw_mymodel_out)

            baseline_output = pack_model_output_for_seddoa52(
                raw_baseline_out,
                n_classes=MODEL_N_CLASSES,
            )   # [B, T, 52]

            mymodel_output = pack_model_output_for_seddoa52(
                raw_mymodel_out,
                n_classes=MODEL_N_CLASSES,
            )   # [B, T, 52]

            baseline_output = baseline_output.detach().cpu().numpy()
            mymodel_output = mymodel_output.detach().cpu().numpy()

            if inspect_first_batch and batch_idx == 0:
                print('baseline_output shape:', baseline_output.shape)
                print('mymodel_output shape :', mymodel_output.shape)
                print('baseline nan count   :', np.isnan(baseline_output).sum())
                print('mymodel nan count    :', np.isnan(mymodel_output).sum())

            for i, wav_name in enumerate(wav_names):
                if not isinstance(wav_name, str):
                    wav_name = str(wav_name)

                wav_name = os.path.splitext(wav_name)[0].strip()

                if wav_name == '':
                    raise ValueError(f'빈 wav_name 발견: wav_names={wav_names}')

                baseline_out_dict = seddoa52_tensor_to_output_dict(
                    baseline_output[i],
                    n_classes=MODEL_N_CLASSES,
                    sed_threshold=sed_threshold,
                    apply_sigmoid_to_sed=apply_sigmoid_to_sed,
                )
                mymodel_out_dict = seddoa52_tensor_to_output_dict(
                    mymodel_output[i],
                    n_classes=MODEL_N_CLASSES,
                    sed_threshold=sed_threshold,
                    apply_sigmoid_to_sed=apply_sigmoid_to_sed,
                )

                write_output_format_file(
                    os.path.join(baseline_pred_dir, f'{wav_name}.csv'),
                    baseline_out_dict
                )
                write_output_format_file(
                    os.path.join(mymodel_pred_dir, f'{wav_name}.csv'),
                    mymodel_out_dict
                )

                if inspect_first_batch and batch_idx == 0 and i == 0:
                    print('saving wav_name:', repr(wav_name))
                    print('baseline frames with events:', len(baseline_out_dict))
                    print('mymodel frames with events :', len(mymodel_out_dict))

                    if len(baseline_out_dict) > 0:
                        fk = sorted(baseline_out_dict.keys())[0]
                        print('baseline first frame/event:', fk, baseline_out_dict[fk][:3])

                    if len(mymodel_out_dict) > 0:
                        fk = sorted(mymodel_out_dict.keys())[0]
                        print('mymodel first frame/event :', fk, mymodel_out_dict[fk][:3])

            if (batch_idx + 1) % 20 == 0 or (batch_idx + 1) == len(loader):
                print(f'  [{batch_idx + 1}/{len(loader)}] done')

    # 6) evaluation
    print('\n>>> SELD metric 평가')
    score_obj = ComputeSELDResults(ref_files_folder=GT_EVAL_ROOT)

    baseline_metrics = score_obj.get_SELD_Results(baseline_pred_dir)
    mymodel_metrics = score_obj.get_SELD_Results(mymodel_pred_dir)

    print_seld_metrics('Baseline', baseline_metrics)
    print_seld_metrics('MyModel', mymodel_metrics)

    return {
        'baseline_pred_dir': baseline_pred_dir,
        'mymodel_pred_dir': mymodel_pred_dir,
        'baseline_metrics': baseline_metrics,
        'mymodel_metrics': mymodel_metrics,
        'gt_eval_root': GT_EVAL_ROOT,
    }


print('dataset / model / evaluate cell ready')

dataset / model / evaluate cell ready


In [78]:
eval_result = evaluate_baseline_vs_mymodel(
    sim_wav_dir=SIM_WAV_DIR,
    label_root_dir=LABEL_ROOT_DIR,
    baseline_ckpt_path=CHECKPOINT_PATH_BASELINE,
    mymodel_ckpt_path=CHECKPOINT_PATH_MY15,
    scaler_path=WTS_PATH,
    batch_size=1,
    num_workers=0,
    output_root=EVAL_OUTPUT_ROOT,
    sed_threshold=0.5,
    apply_sigmoid_to_sed=False,
    inspect_first_batch=True,
)

eval_result

[OK] no-distance GT 생성 완료: expand_features\labels_eval_nodist
[Dataset] matched items: 50
[Dataset] first few stems: ['fold4_room24_mix000', 'fold4_room24_mix001', 'fold4_room24_mix002', 'fold4_room24_mix003', 'fold4_room24_mix004']
[OK] baseline ckpt loaded: c:/26'interspeech/Solution_on_3D_SELD/output/SED-DOA_FOA/checkpoints/checkpoint_epoch357_step97461.pth
[OK] my_model ckpt loaded: c:/26'interspeech/Monaural_SELD/output/my_experiment15/checkpoints/checkpoint_epoch337_step92001.pth

>>> inference + prediction csv 저장 시작

[baseline raw output] type = <class 'torch.Tensor'>
  tensor shape: (1, 100, 52)

[my_model raw output] type = <class 'torch.Tensor'>
  tensor shape: (1, 100, 52)
baseline_output shape: (1, 100, 52)
mymodel_output shape : (1, 100, 52)
baseline nan count   : 0
mymodel nan count    : 0
saving wav_name: 'fold4_room24_mix000'
baseline frames with events: 91
mymodel frames with events : 100
baseline first frame/event: 0 [[1, np.float32(0.69224346), np.float32(0.66245425)

{'baseline_pred_dir': 'expand_features\\eval_compare\\baseline_preds',
 'mymodel_pred_dir': 'expand_features\\eval_compare\\mymodel_preds',
 'baseline_metrics': (np.float64(1.174151149901457),
  np.float64(0.013011180272781912),
  np.float64(128.90660017063186),
  np.float64(0.11429021384906365),
  np.float64(0.9407493836263361),
  array([[1.17415115e+00, 1.17415115e+00, 1.17415115e+00, 1.17415115e+00,
          1.17415115e+00, 1.17415115e+00, 1.17415115e+00, 1.17415115e+00,
          1.17415115e+00, 1.17415115e+00, 1.17415115e+00, 1.17415115e+00,
          1.17415115e+00],
         [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 1.19047619e-02,
          8.24742266e-02, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
          4.67289719e-02, 2.80373831e-02, 0.00000000e+00, 0.00000000e+00,
          0.00000000e+00],
         [1.80000000e+02, 5.27668593e+01, 1.80000000e+02, 9.74025814e+01,
          4.32861248e+01, 1.13888860e+02, 1.80000000e+02, 1.80000000e+02,
          7.66645025e+

## 4. 체크포인트 로딩 및 AuditoryFrontend 래퍼

my_model15 / baseline 두 모델을 각자의 체크포인트로 로드합니다.

In [65]:
# =============================================================================
# Dual-model evaluation on simulated SELD dataset
# baseline  : log-mel + gcc + scaler.joblib -> ResNet-Conformer
# my_model  : raw waveform -> frontend 포함 모델
# =============================================================================

import os    
import shutil
import glob
import joblib
import soundfile as sf
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

from models.my_model15 import (
    ResnetConformer_seddoa_nopool_2023 as MyModel15,
)
from models.resnet_conformer_audio import (
    ResnetConformer_seddoa_nopool_2023 as BaselineModel,
)

from utils.sed_doa import SedDoaResult_2023
from utils.write_csv import write_output_format_file
from utils.cls_tools.cls_compute_seld_results import ComputeSELDResults
from utils.cls_tools.cls_feature_class import FeatureClass


# ---------------------------------------------------------------------
# 0. 공통 설정
# ---------------------------------------------------------------------
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# checkpoint 구조 기준 클래스 수
# 현재 notebook 상단 로딩 코드와 맞추려면 out_dim=39 유지
# (= 13 classes * 3)
N_CLASSES_CKPT = 13
OUT_DIM = 39

EVAL_OUTPUT_ROOT = os.path.join(NORM_PATH, 'eval_compare')
BASELINE_PRED_DIR = os.path.join(EVAL_OUTPUT_ROOT, 'baseline_preds')
MYMODEL_PRED_DIR  = os.path.join(EVAL_OUTPUT_ROOT, 'mymodel_preds')

os.makedirs(EVAL_OUTPUT_ROOT, exist_ok=True)
os.makedirs(BASELINE_PRED_DIR, exist_ok=True)
os.makedirs(MYMODEL_PRED_DIR, exist_ok=True)


# ---------------------------------------------------------------------
# 1. helper
# ---------------------------------------------------------------------
def load_ckpt_state(ckpt_path, map_location='cpu'):
    ckpt = torch.load(ckpt_path, map_location=map_location)
    state = ckpt.get('model_state_dict', ckpt)

    # DataParallel prefix 제거
    new_state = {}
    for k, v in state.items():
        if k.startswith('module.'):
            new_state[k[len('module.'):]] = v
        else:
            new_state[k] = v
    return new_state


def pad_or_crop_multich_wav(x, target_len):
    """
    x: [C, T]
    """
    if x.shape[1] > target_len:
        x = x[:, :target_len]
    elif x.shape[1] < target_len:
        pad_len = target_len - x.shape[1]
        x = np.pad(x, ((0, 0), (0, pad_len)), mode='constant')
    return x


def load_multich_wav(wav_path, target_sr=SR, target_len=SCENE_T):
    """
    저장된 sim wav 읽기
    저장 시 scene.T 로 썼으므로 보통 [T, C]로 읽힘 -> [C, T]로 transpose
    """
    x, sr = sf.read(wav_path)
    if sr != target_sr:
        raise ValueError(f"샘플레이트 불일치: {wav_path}, got={sr}, expected={target_sr}")

    x = np.asarray(x, dtype=np.float32)

    if x.ndim == 1:
        raise ValueError(f"multichannel wav가 아닙니다: {wav_path}, shape={x.shape}")

    # [T, C] -> [C, T]
    if x.shape[0] > x.shape[1]:
        x = x.T

    if x.shape[0] != N_MICS:
        raise ValueError(f"마이크 채널 수 불일치: {wav_path}, got={x.shape[0]}, expected={N_MICS}")

    x = pad_or_crop_multich_wav(x, target_len)
    return x


def build_feature_extractor():
    """
    baseline log-mel + gcc 추출용
    """
    params = {
        'dataset_dir': SIM_WAV_DIR,
        'feat_label_dir': NORM_PATH,
        'dataset': 'mic',
        'fs': SR,
        'hop_len_s': HOP / SR,
        'label_hop_len_s': LABEL_HOP_S,
        'nb_mel_bins': N_MELS,
        'unique_classes': N_CLASSES_CKPT,
        'spatial_map': False,
        'multi_accdoa': False,
        'use_salsalite': False,
        'raw_chunks': False,
        'saved_chunks': False,
    }
    return FeatureClass(params, is_eval=True)


def baseline_preprocess_from_waveform(mic_signals, feat_cls, scaler):
    """
    mic_signals: [4, T] numpy
    return: [10, frames, 64] torch.float32
    """
    audio_in = mic_signals.T.astype(np.float32)  # [T, 4]
    audio_in = audio_in + feat_cls._eps

    nb_feat_frames = int(mic_signals.shape[1] / float(feat_cls._hop_len))

    spectra = feat_cls._spectrogram(audio_in, nb_feat_frames)
    mel_spect = feat_cls._get_mel_spectrogram(spectra)   # expected: [frames, 4*64]
    gcc = feat_cls._get_gcc(spectra)                     # expected: [frames, 6*64]

    feat = np.concatenate((mel_spect, gcc), axis=-1)    # [frames, 10*64]

    expected_dim = N_CHANNELS * N_MELS
    assert feat.shape[1] == expected_dim, \
        f"baseline feature dim mismatch: got {feat.shape[1]}, expected {expected_dim}"

    feat = scaler.transform(feat)                        # [frames, 640]
    feat = feat.reshape(feat.shape[0], N_CHANNELS, N_MELS)  # [frames, 10, 64]
    feat = np.transpose(feat, (1, 0, 2))                # [10, frames, 64]

    return torch.from_numpy(feat.astype(np.float32))


def raw_preprocess_for_mymodel(mic_signals):
    """
    mic_signals: [4, T] numpy
    return: [4, n_frames, HOP] torch.float32

    my_model(frontend 포함)이 받는 raw frame tensor
    """
    T = mic_signals.shape[1]
    n_frames = T // HOP
    usable = n_frames * HOP
    wav = mic_signals[:, :usable].reshape(N_MICS, n_frames, HOP)
    return torch.from_numpy(wav.astype(np.float32))


def maybe_unwrap_model_output(output):
    """
    모델 forward 반환형이 tuple/list/dict일 때 첫 prediction tensor를 꺼내기 위한 helper.
    네 코드베이스의 forward가 이미 tensor 하나만 반환하면 그대로 통과.
    """
    if isinstance(output, torch.Tensor):
        return output

    if isinstance(output, (list, tuple)):
        return output[0]

    if isinstance(output, dict):
        for k in ['output', 'pred', 'accdoa', 'seddoa', 'logits']:
            if k in output:
                return output[k]
        # fallback
        return next(iter(output.values()))

    return output


def print_seld_metrics(tag, metrics_tuple):
    ER, F, LE, LR, seld_scr, classwise = metrics_tuple
    print(f'\n[{tag}]')
    print(f'ER        : {ER:.4f}')
    print(f'F-score   : {F:.4f}')
    print(f'LE        : {LE:.4f}')
    print(f'LR        : {LR:.4f}')
    print(f'SELD score: {seld_scr:.4f}')


# ---------------------------------------------------------------------
# 2. Dataset
# ---------------------------------------------------------------------
class SimulatedSELDComparisonDataset(Dataset):
    """
    sim_wavs/*.wav 와 labels/*.csv 의 stem을 맞춰서 읽음
    baseline_input: [10, frames, 64]
    raw_input     : [4, n_frames, HOP]
    wav_name      : stem (예: sim_0000)
    """
    def __init__(self, sim_wav_dir, label_dir, scaler_path):
        self.sim_wav_dir = sim_wav_dir
        self.label_dir = label_dir

        self.feat_cls = build_feature_extractor()
        self.scaler = joblib.load(scaler_path)

        wav_map = {
            os.path.splitext(os.path.basename(p))[0]: p
            for p in glob.glob(os.path.join(sim_wav_dir, '**', '*.wav'), recursive=True)
        }
        label_map = {
            os.path.splitext(os.path.basename(p))[0]: p
            for p in glob.glob(os.path.join(label_dir, '**', '*.csv'), recursive=True)
        }

        common_stems = sorted(set(wav_map.keys()) & set(label_map.keys()))
        if not common_stems:
            raise ValueError(
                f"공통 stem이 없습니다.\n"
                f"sim_wav_dir={sim_wav_dir}\n"
                f"label_dir={label_dir}"
            )

        self.items = [(stem, wav_map[stem], label_map[stem]) for stem in common_stems]
        print(f'[Dataset] matched items: {len(self.items)}')

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        stem, wav_path, label_path = self.items[idx]

        mic_signals = load_multich_wav(
            wav_path,
            target_sr=SR,
            target_len=SCENE_T,
        )  # [4, T]

        baseline_input = baseline_preprocess_from_waveform(
            mic_signals=mic_signals,
            feat_cls=self.feat_cls,
            scaler=self.scaler,
        )  # [10, frames, 64]

        raw_input = raw_preprocess_for_mymodel(
            mic_signals=mic_signals
        )  # [4, n_frames, HOP]

        return {
            'wav_name': stem,
            'wav_path': wav_path,
            'label_path': label_path,
            'baseline_input': baseline_input,
            'raw_input': raw_input,
        }


# ---------------------------------------------------------------------
# 3. 모델 로딩
# ---------------------------------------------------------------------
def build_models_for_comparison(
    baseline_ckpt_path,
    mymodel_ckpt_path,
    norm_path,
    device=DEVICE,
):
    baseline_model = BaselineModel(
        in_channel=N_CHANNELS,
        in_dim=N_MELS,
        out_dim=OUT_DIM,
    ).to(device)

    my_model = MyModel15(
        in_channel=N_CHANNELS,
        in_dim=N_MELS,
        out_dim=OUT_DIM,
        norm_path=norm_path,
    ).to(device)

    baseline_state = load_ckpt_state(baseline_ckpt_path, map_location=device)
    mymodel_state = load_ckpt_state(mymodel_ckpt_path, map_location=device)

    baseline_model.load_state_dict(baseline_state, strict=True)
    my_model.load_state_dict(mymodel_state, strict=True)

    baseline_model.eval()
    my_model.eval()

    print(f'[OK] baseline ckpt loaded: {baseline_ckpt_path}')
    print(f'[OK] my_model ckpt loaded: {mymodel_ckpt_path}')

    return baseline_model, my_model


# ---------------------------------------------------------------------
# 4. 메인 평가 함수
# ---------------------------------------------------------------------
def evaluate_baseline_vs_mymodel(
    sim_wav_dir=SIM_WAV_DIR,
    ref_csv_dir=LABEL_DIR,
    baseline_ckpt_path=CHECKPOINT_PATH_BASELINE,
    mymodel_ckpt_path=CHECKPOINT_PATH_MY15,
    scaler_path=WTS_PATH,
    batch_size=1,
    num_workers=0,
    segment_length=100,
    output_root=EVAL_OUTPUT_ROOT,
):
    dataset = SimulatedSELDComparisonDataset(
        sim_wav_dir=sim_wav_dir,
        label_dir=ref_csv_dir,
        scaler_path=scaler_path,
    )
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )

    baseline_model, my_model = build_models_for_comparison(
        baseline_ckpt_path=baseline_ckpt_path,
        mymodel_ckpt_path=mymodel_ckpt_path,
        norm_path=scaler_path,
        device=DEVICE,
    )

    baseline_result = SedDoaResult_2023(segment_length=segment_length)
    mymodel_result = SedDoaResult_2023(segment_length=segment_length)

    print('\n>>> inference 시작')
    with torch.no_grad():
        for batch_idx, batch in enumerate(loader):
            wav_names = batch['wav_name']   # list[str]

            fake_wav_names = [f"{name}_dummy_dummy_0" for name in wav_names]
            baseline_input = batch['baseline_input'].to(DEVICE)   # [B, 10, frames, 64]
            raw_input = batch['raw_input'].to(DEVICE)             # [B, 4, n_frames, HOP]

            baseline_output = maybe_unwrap_model_output(
                baseline_model(baseline_input)
            )
            mymodel_output = maybe_unwrap_model_output(
                my_model(raw_input)
            )

            baseline_result.add_items(fake_wav_names, baseline_output)
            mymodel_result.add_items(fake_wav_names, mymodel_output)

            if (batch_idx + 1) % 20 == 0 or (batch_idx + 1) == len(loader):
                print(f'  [{batch_idx + 1}/{len(loader)}] done')

    baseline_pred_dir = os.path.join(output_root, 'baseline_preds')
    mymodel_pred_dir = os.path.join(output_root, 'mymodel_preds')

    for d in [baseline_pred_dir, mymodel_pred_dir]:
        if os.path.exists(d):
            shutil.rmtree(d)
        os.makedirs(d, exist_ok=True)

    print('\n>>> prediction csv 저장')
    for csv_name, out_dict in baseline_result.get_result().items():
        write_output_format_file(
            os.path.join(baseline_pred_dir, '{}.csv'.format(csv_name)),
            out_dict
        )

    for csv_name, out_dict in mymodel_result.get_result().items():
        write_output_format_file(
            os.path.join(mymodel_pred_dir, '{}.csv'.format(csv_name)),
            out_dict
        )
    print("saving:", repr(csv_name))
    print('\n>>> SELD metric 평가')
    score_obj = ComputeSELDResults(ref_files_folder=ref_csv_dir)

    baseline_metrics = score_obj.get_SELD_Results(baseline_pred_dir)
    mymodel_metrics = score_obj.get_SELD_Results(mymodel_pred_dir)
    
    print_seld_metrics('Baseline', baseline_metrics)
    print_seld_metrics('MyModel', mymodel_metrics)

    return {
        'baseline_pred_dir': baseline_pred_dir,
        'mymodel_pred_dir': mymodel_pred_dir,
        'baseline_metrics': baseline_metrics,
        'mymodel_metrics': mymodel_metrics,
    }

In [66]:
# 실제 실행
eval_result = evaluate_baseline_vs_mymodel(
    sim_wav_dir=SIM_WAV_DIR,
    ref_csv_dir=LABEL_DIR,
    baseline_ckpt_path=CHECKPOINT_PATH_BASELINE,
    mymodel_ckpt_path=CHECKPOINT_PATH_MY15,
    scaler_path=WTS_PATH,
    batch_size=1,
    num_workers=0,
    segment_length=100,
    output_root=EVAL_OUTPUT_ROOT,
)

eval_result

[Dataset] matched items: 250
[OK] baseline ckpt loaded: c:\26'interspeech\Solution_on_3D_SELD/output/SED-DOA_FOA/checkpoints/checkpoint_epoch357_step97461.pth
[OK] my_model ckpt loaded: c:\26'interspeech\Monaural_SELD/output/my_experiment15/checkpoints/checkpoint_epoch337_step92001.pth

>>> inference 시작
  [20/250] done
  [40/250] done
  [60/250] done
  [80/250] done
  [100/250] done
  [120/250] done
  [140/250] done
  [160/250] done
  [180/250] done
  [200/250] done
  [220/250] done
  [240/250] done
  [250/250] done

>>> prediction csv 저장
saving: 'sim_0249'

>>> SELD metric 평가

[Baseline]
ER        : 3.6494
F-score   : 0.0247
LE        : 120.6129
LR        : 0.1867
SELD score: 1.5270

[MyModel]
ER        : 3.4259
F-score   : 0.0365
LE        : 98.2098
LR        : 0.1746
SELD score: 1.4401


{'baseline_pred_dir': 'expand_features\\eval_compare\\baseline_preds',
 'mymodel_pred_dir': 'expand_features\\eval_compare\\mymodel_preds',
 'baseline_metrics': (np.float64(3.6494117643646717),
  np.float64(0.024705839932450863),
  np.float64(120.61291216599024),
  np.float64(0.18666247416884263),
  np.float64(1.52702879612972),
  array([[3.64941176e+00, 3.64941176e+00, 3.64941176e+00, 3.64941176e+00,
          3.64941176e+00, 3.64941176e+00, 3.64941176e+00, 3.64941176e+00,
          3.64941176e+00, 3.64941176e+00, 3.64941176e+00, 3.64941176e+00,
          3.64941176e+00],
         [3.98009950e-02, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
          0.00000000e+00, 1.40024783e-01, 0.00000000e+00, 0.00000000e+00,
          0.00000000e+00, 7.38310090e-02, 6.45161290e-02, 0.00000000e+00,
          3.00300300e-03],
         [2.10040276e+01, 1.80000000e+02, 1.51042484e+02, 1.80000000e+02,
          4.49267871e+01, 4.33646464e+01, 1.80000000e+02, 1.80000000e+02,
          1.80000000e+0